# Análisis integral CRISP-DM — cartera mensual

**Fuente:** `DATOS22.csv`, separado por `;`, sin encabezado.

Este notebook analiza un corte mensual de cartera con seis fases CRISP-DM:

1. Comprensión del negocio.
2. Comprensión de los datos.
3. Preparación de los datos.
4. Modelado diagnóstico exploratorio.
5. Evaluación.
6. Entrega de resultados.

El análisis muestra tablas de cantidades y porcentajes, guarda resultados en
CSV y genera gráficos con fucsia, magenta, lila, azul y blanco.

> **Alcance temporal:** con un solo mes no se puede validar un modelo que
> prediga mora futura. El modelo incluido clasifica mora del mismo corte y
> sirve únicamente como diagnóstico técnico. Para producción se necesitan
> varios meses y una variable objetivo observada en T+1 o después.



In [ ]:
from pathlib import Path
from IPython.display import display, Markdown
from matplotlib.colors import LinearSegmentedColormap
import json
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")

FUCSIA = "#FF2DAA"
MAGENTA = "#C218A5"
LILA = "#9B5DE5"
AZUL = "#246BFD"
AZUL_OSCURO = "#14213D"
BLANCO = "#FFFFFF"
GRIS = "#F3F1F8"
PALETA = [FUCSIA, MAGENTA, LILA, AZUL, AZUL_OSCURO]
CMAP = LinearSegmentedColormap.from_list("fucsia_lila_azul", [BLANCO, LILA, FUCSIA, AZUL])

plt.rcParams.update(
    {
        "figure.figsize": (12, 6),
        "figure.facecolor": BLANCO,
        "axes.facecolor": BLANCO,
        "axes.edgecolor": AZUL_OSCURO,
        "axes.titleweight": "bold",
        "axes.titlesize": 15,
        "axes.labelcolor": AZUL_OSCURO,
        "text.color": AZUL_OSCURO,
        "xtick.color": AZUL_OSCURO,
        "ytick.color": AZUL_OSCURO,
        "grid.color": "#DDD8EA",
        "grid.alpha": 0.55,
    }
)

ARCHIVO_CSV = Path("DATOS22.csv")
CARPETA_SALIDA = Path("salidas_crisp_dm")
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
EXPORTAR_DATOS_LIMPIOS = False
EJECUTAR_MODELO_DIAGNOSTICO = True
SEMILLA = 42


def guardar_figura(nombre: str) -> None:
    plt.tight_layout()
    plt.savefig(CARPETA_SALIDA / nombre, dpi=160, bbox_inches="tight", facecolor=BLANCO)
    plt.show()


def estilo_tabla(tabla: pd.DataFrame, gradiente: str | None = None):
    estilos = [
        {"selector": "th", "props": [("background-color", AZUL_OSCURO), ("color", BLANCO)]},
        {"selector": "td", "props": [("background-color", BLANCO), ("color", AZUL_OSCURO)]},
        {"selector": "caption", "props": [("color", MAGENTA), ("font-size", "15px"), ("font-weight", "bold")]},
    ]
    resultado = tabla.style.set_table_styles(estilos).format(precision=2, thousands=",")
    if gradiente and gradiente in tabla.columns:
        resultado = resultado.background_gradient(cmap=CMAP, subset=[gradiente])
    return resultado


def grafico_barras(serie: pd.Series, titulo: str, nombre: str, horizontal: bool = True) -> None:
    serie = serie.dropna()
    colores = [PALETA[indice % len(PALETA)] for indice in range(len(serie))]
    fig, ax = plt.subplots(figsize=(12, max(5, len(serie) * 0.42)))
    if horizontal:
        barras = ax.barh(serie.index.astype(str), serie.values, color=colores)
        ax.invert_yaxis()
        ax.set_xlabel("Cantidad")
        ax.bar_label(barras, fmt="{:,.0f}", padding=3, fontsize=8)
    else:
        barras = ax.bar(serie.index.astype(str), serie.values, color=colores)
        ax.tick_params(axis="x", rotation=35)
        ax.set_ylabel("Cantidad")
        ax.bar_label(barras, fmt="{:,.0f}", padding=3, fontsize=8)
    ax.set_title(titulo)
    ax.grid(axis="x" if horizontal else "y")
    guardar_figura(nombre)




## 1. Comprensión del negocio

El objetivo es describir la cartera del mes, medir su calidad de datos y
responder preguntas útiles para la gestión:

- ¿Cuántos créditos y clientes contiene el corte?
- ¿Qué columnas presentan faltantes y en qué proporción?
- ¿Existen duplicados, constantes, valores imposibles o extremos?
- ¿Cómo se distribuyen saldo, edad, plazo, transacciones y mora?
- ¿Qué segmentos concentran saldo y mora de 30 días o más?
- ¿Qué fuentes opcionales tienen cobertura suficiente?
- ¿Qué información falta para construir un modelo de mora futura?



## 2. Comprensión de los datos — definición de columnas



In [ ]:
COLUMNAS = [
    "FechaCorte", "CarteraKey", "ClienteKey", "EdadCliente", "GeneroCliente",
    "EstadoCivilCliente", "NumeroHijosCliente", "CiudadCliente", "DepartamentoCliente",
    "BarrioCliente", "AntiguedadClienteMeses", "SegmentoCliente", "NacionalidadCliente",
    "EsJefaHogar", "NivelIngresoCliente", "NivelEgresoCliente", "FuenteIngresoCliente",
    "ProfesionCliente", "GradoInstruccionCliente", "PatrimonioClienteBs", "ActivosClienteBs",
    "VentasAnualesClienteBs", "InfocredPeorCalificacionASFIKey", "InfocredSaldoVigenteBs",
    "InfocredSaldoVencidoBs", "InfocredSaldoEjecucionBs", "InfocredSaldoCastigadoBs",
    "InfocredSaldoContingenteBs", "InfocredMesesHastaMaxVencimiento",
    "InfocredMontoDeudaTotalBs", "InfocredCuotaBs", "InfocredCantDeudasOtrasEntidades",
    "InfocredCantDeudasDeudor", "InfocredCantDeudasCodeudor", "InfocredEsCreditoVivienda",
    "ProductoCartera", "EsDeudorCartera", "EsCodeudorCartera", "EsGaranteCartera",
    "EsClienteDigital", "EsClienteTinka", "CodigoOficialCredito", "NombreOficialCredito",
    "TipoCredito", "TipoBanca", "TasaEfectivaCredito", "TasaBaseCredito", "TasaTRECredito",
    "PlazoCreditoDias", "PlazoCreditoMeses", "FrecuenciaPagoCredito", "TipoAmortizacion",
    "CantidadReprogramaciones", "MontoReprogramadoBs", "EstadoCredito",
    "SectorEconomicoCliente", "SectorEconomicoCredito", "ActividadEconomicaCliente",
    "ActividadEconomicaCredito", "ActividadEconomicaClienteCAEDEC",
    "ActividadEconomicaCreditoCAEDEC", "NumeroTrabajadores", "IndiceActividadEconomica",
    "TamanioEmpresa", "SaldoCreditoBs", "SaldoCapitalBs", "SaldoDiferidoBs", "InteresBs",
    "OtrosRubrosBs", "MontoDesembolsadoBs", "GarantiaHipotecariaBs", "GarantiaPersonalBs",
    "GarantiaPrendariaBs", "GarantiaCaptacionBs", "GarantiaOtrosBs",
    "GarantiaDocumentosCustodiaBs", "MontoGarantiaTotalBs", "PrevisionRequeridaBs",
    "PrevisionCiclicaBs", "MesesHastaVencimientoCredito",
    "MesesHastaVencimientoReprogramado", "MesesDiferenciaRefinanciamiento",
    "DiasHastaProximoPago", "Evento", "EventoDuracionDias", "EventoCodigoRegional",
    "EventoCodigoDepartamento", "EventoDescripcion", "MaxDiasMoraMes", "MaxDiasMora3M",
    "PromedioMora3M", "MesesConMora3M", "MaxDiasMora6M", "PromedioMora6M",
    "MesesConMora6M", "MaxDiasMora12M", "PromedioMora12M", "MesesConMora12M",
    "TasaCumplimiento", "AntiguedadCreditoMeses", "CalificacionFIE", "CalificacionASFI",
    "CantidadTrx", "MontoTotalTrxBs", "CantidadDepositos", "CantidadRetiros",
    "MontoDepositosBs", "MontoRetirosBs", "FlujoNetoBs", "CantidadQR", "CantidadDigital",
    "CantidadPresencial", "CantidadAutoservicio", "CantidadProcesamiento", "MontoQRBs",
    "MontoDigitalBs", "MontoPresencialBs", "MontoAutoservicioBs", "MontoProcesamientoBs",
    "CantidadTrx3M", "MontoTotalBs3M", "CantidadDepositos3M", "CantidadRetiros3M",
    "MontoDepositosBs3M", "MontoRetirosBs3M", "CantidadTrx6M", "MontoTotalBs6M",
    "CantidadDepositos6M", "CantidadRetiros6M", "MontoDepositosBs6M", "MontoRetirosBs6M",
    "CantidadTrx12M", "MontoTotalBs12M", "CantidadDepositos12M", "CantidadRetiros12M",
    "MontoDepositosBs12M", "MontoRetirosBs12M", "MesesConHistoria",
    "TieneTransaccionesMes", "TieneHistoricoTransaccional", "TieneInfocred", "TieneEvento",
]

CATEGORICAS = [
    "GeneroCliente", "EstadoCivilCliente", "CiudadCliente", "DepartamentoCliente",
    "BarrioCliente", "SegmentoCliente", "NacionalidadCliente", "EsJefaHogar",
    "NivelIngresoCliente", "NivelEgresoCliente", "FuenteIngresoCliente", "ProfesionCliente",
    "GradoInstruccionCliente", "InfocredEsCreditoVivienda", "ProductoCartera",
    "EsDeudorCartera", "EsCodeudorCartera", "EsGaranteCartera", "EsClienteDigital",
    "EsClienteTinka", "NombreOficialCredito", "TipoCredito", "TipoBanca",
    "FrecuenciaPagoCredito", "TipoAmortizacion", "EstadoCredito", "SectorEconomicoCliente",
    "SectorEconomicoCredito", "ActividadEconomicaCliente", "ActividadEconomicaCredito",
    "ActividadEconomicaClienteCAEDEC", "ActividadEconomicaCreditoCAEDEC", "TamanioEmpresa",
    "Evento", "EventoDescripcion", "CalificacionFIE", "CalificacionASFI",
]

IDENTIFICADORES = ["CarteraKey", "ClienteKey", "CodigoOficialCredito"]
INDICADORES_COBERTURA = [
    "TieneTransaccionesMes", "TieneHistoricoTransaccional", "TieneInfocred", "TieneEvento"
]

print(f"Columnas esperadas: {len(COLUMNAS)}")
assert len(COLUMNAS) == len(set(COLUMNAS)), "Hay nombres de columnas repetidos."



### Carga robusta del CSV

La celda prueba codificaciones habituales, reconoce valores vacíos y valida
que el archivo corresponda exactamente a la estructura esperada.



In [ ]:
def localizar_csv(nombre: Path) -> Path:
    candidatos = [nombre, Path.cwd() / nombre.name, Path.cwd().parent / nombre.name]
    for candidato in candidatos:
        if candidato.exists():
            return candidato.resolve()
    encontrados = list(Path.cwd().rglob(nombre.name))
    if encontrados:
        return encontrados[0].resolve()
    raise FileNotFoundError(
        f"No se encontró {nombre.name}. Coloque el CSV junto al notebook. "
        f"Carpeta actual: {Path.cwd()}"
    )


def leer_datos(ruta: Path) -> tuple[pd.DataFrame, str]:
    ultimo_error = None
    for codificacion in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            tabla = pd.read_csv(
                ruta,
                header=None,
                names=COLUMNAS,
                sep=";",
                encoding=codificacion,
                quotechar='"',
                na_values=["", " ", "NULL", "null", "None", "N/A", "NA"],
                keep_default_na=True,
                low_memory=False,
                index_col=False,
                on_bad_lines="error",
            )
            return tabla, codificacion
        except UnicodeDecodeError as error:
            ultimo_error = error
    raise UnicodeError(f"No se pudo identificar la codificación: {ultimo_error}")


ruta_csv = localizar_csv(ARCHIVO_CSV)
datos, codificacion_usada = leer_datos(ruta_csv)

if not datos.empty and str(datos.iloc[0]["FechaCorte"]).strip() == "FechaCorte":
    datos = datos.iloc[1:].reset_index(drop=True)

if list(datos.columns) != COLUMNAS:
    raise ValueError("La estructura leída no coincide con las columnas definidas.")

print(f"Archivo: {ruta_csv}")
print(f"Codificación: {codificacion_usada}")
print(f"Dimensión inicial: {datos.shape[0]:,} filas × {datos.shape[1]} columnas")
display(datos.head(3))



## 3. Preparación de los datos

Se normalizan blancos, fechas, números y texto posiblemente mal codificado.
No se imputan valores todavía: primero se conserva y mide el faltante real.



In [ ]:
def reparar_texto(valor):
    if not isinstance(valor, str) or not any(marca in valor for marca in ("Ã", "Â", "â")):
        return valor
    try:
        return valor.encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return valor


for columna in CATEGORICAS:
    serie = datos[columna].astype("string").str.strip()
    serie = serie.replace({"": pd.NA, "<NA>": pd.NA})
    mascara = serie.str.contains(r"Ã|Â|â", regex=True, na=False)
    if mascara.any():
        serie.loc[mascara] = serie.loc[mascara].map(reparar_texto)
    datos[columna] = serie

datos["FechaCorte"] = pd.to_datetime(datos["FechaCorte"], errors="coerce")

columnas_numericas = [
    columna for columna in COLUMNAS
    if columna not in CATEGORICAS and columna != "FechaCorte"
]
for columna in columnas_numericas:
    datos[columna] = pd.to_numeric(datos[columna], errors="coerce")

periodos = datos["FechaCorte"].dropna().dt.to_period("M").unique()
if len(periodos) != 1:
    raise ValueError(f"Se esperaba un solo mes y se encontraron: {list(map(str, periodos))}")

datos["Mora30"] = pd.Series(pd.NA, index=datos.index, dtype="Int8")
mascara_objetivo = datos["MaxDiasMoraMes"].notna()
datos.loc[mascara_objetivo, "Mora30"] = (
    datos.loc[mascara_objetivo, "MaxDiasMoraMes"] >= 30
).astype("int8")

bandas_mora = [-np.inf, 0, 29, 59, 89, np.inf]
etiquetas_mora = ["Al día", "1–29 días", "30–59 días", "60–89 días", "90+ días"]
datos["BandaMora"] = pd.cut(
    datos["MaxDiasMoraMes"], bins=bandas_mora, labels=etiquetas_mora
).astype("string").fillna("Sin dato")

memoria_mb = datos.memory_usage(deep=True).sum() / 1024**2
print(f"Corte analizado: {periodos[0]}")
print(f"Memoria aproximada: {memoria_mb:,.1f} MB")



## 2.1 Resumen general y granularidad



In [ ]:
duplicados_fila = int(datos.duplicated().sum())
duplicados_clave = int(datos.duplicated(["FechaCorte", "CarteraKey"], keep=False).sum())

resumen_general = pd.DataFrame(
    {
        "Indicador": [
            "Mes", "Filas", "Columnas", "Carteras únicas", "Clientes únicos",
            "Duplicados exactos", "Filas con clave FechaCorte-CarteraKey repetida",
            "Memoria MB",
        ],
        "Valor": [
            str(periodos[0]), f"{len(datos):,}", len(datos.columns),
            f"{datos['CarteraKey'].nunique(dropna=True):,}",
            f"{datos['ClienteKey'].nunique(dropna=True):,}",
            f"{duplicados_fila:,}", f"{duplicados_clave:,}", f"{memoria_mb:,.1f}",
        ],
    }
)
display(estilo_tabla(resumen_general).set_caption("Resumen del dataset mensual"))



## 2.2 Faltantes por cada columna



In [ ]:
tabla_faltantes = pd.DataFrame(
    {
        "Columna": datos.columns,
        "TipoDato": datos.dtypes.astype(str).values,
        "CantidadFaltante": datos.isna().sum().values,
        "CantidadDisponible": datos.notna().sum().values,
        "ValoresUnicos": [datos[c].nunique(dropna=True) for c in datos.columns],
    }
)
tabla_faltantes["PorcentajeFaltante"] = (
    100 * tabla_faltantes["CantidadFaltante"] / len(datos)
).round(2)
tabla_faltantes["PorcentajeDisponible"] = (100 - tabla_faltantes["PorcentajeFaltante"]).round(2)
tabla_faltantes = tabla_faltantes.sort_values(
    ["PorcentajeFaltante", "CantidadFaltante"], ascending=False
).reset_index(drop=True)

display(
    estilo_tabla(tabla_faltantes, "PorcentajeFaltante")
    .set_caption("Cantidad y porcentaje de faltantes — todas las columnas")
)
tabla_faltantes.to_csv(CARPETA_SALIDA / "01_faltantes_por_columna.csv", sep=";", index=False)



In [ ]:
faltantes_grafico = tabla_faltantes.query("CantidadFaltante > 0").head(30).sort_values("PorcentajeFaltante")
fig, ax = plt.subplots(figsize=(12, max(6, len(faltantes_grafico) * 0.35)))
barras = ax.barh(
    faltantes_grafico["Columna"], faltantes_grafico["PorcentajeFaltante"], color=FUCSIA
)
ax.bar_label(barras, labels=[f"{v:.1f}%" for v in faltantes_grafico["PorcentajeFaltante"]], padding=3)
ax.set_title("Top 30 columnas con mayor porcentaje de faltantes")
ax.set_xlabel("Porcentaje faltante")
ax.set_xlim(0, 105)
ax.grid(axis="x")
guardar_figura("01_faltantes_top30.png")



In [ ]:
muestra_nulos = datos[tabla_faltantes.head(30)["Columna"]].isna().head(500).T
fig, ax = plt.subplots(figsize=(14, 9))
ax.imshow(muestra_nulos, aspect="auto", interpolation="nearest", cmap=CMAP)
ax.set_yticks(range(len(muestra_nulos.index)))
ax.set_yticklabels(muestra_nulos.index, fontsize=8)
ax.set_xlabel("Primeras 500 filas")
ax.set_title("Patrón de faltantes — blanco: disponible; color: faltante")
guardar_figura("02_matriz_faltantes.png")



## 2.3 Constantes, duplicados y reglas de calidad



In [ ]:
columnas_constantes = [c for c in datos.columns if datos[c].nunique(dropna=False) <= 1]
columnas_vacias = [c for c in datos.columns if datos[c].isna().all()]
columnas_casi_constantes = []
for columna in datos.columns:
    frecuencias = datos[columna].value_counts(dropna=False, normalize=True)
    if len(frecuencias) > 1 and frecuencias.iloc[0] >= 0.99:
        columnas_casi_constantes.append((columna, frecuencias.iloc[0] * 100))

reglas_calidad = pd.DataFrame(
    {
        "Regla": [
            "Edad menor a 18", "Edad mayor a 100", "Saldo de crédito negativo",
            "Monto desembolsado no positivo", "Tasa efectiva negativa",
            "Mora mensual negativa", "CarteraKey nulo", "ClienteKey nulo",
        ],
        "Cantidad": [
            int((datos["EdadCliente"] < 18).sum()),
            int((datos["EdadCliente"] > 100).sum()),
            int((datos["SaldoCreditoBs"] < 0).sum()),
            int((datos["MontoDesembolsadoBs"] <= 0).sum()),
            int((datos["TasaEfectivaCredito"] < 0).sum()),
            int((datos["MaxDiasMoraMes"] < 0).sum()),
            int(datos["CarteraKey"].isna().sum()),
            int(datos["ClienteKey"].isna().sum()),
        ],
    }
)
reglas_calidad["Porcentaje"] = (100 * reglas_calidad["Cantidad"] / len(datos)).round(4)

display(Markdown(f"**Columnas completamente vacías ({len(columnas_vacias)}):** {columnas_vacias}"))
display(Markdown(f"**Columnas constantes ({len(columnas_constantes)}):** {columnas_constantes}"))
display(
    estilo_tabla(pd.DataFrame(columnas_casi_constantes, columns=["Columna", "PorcentajeModa"]))
    .set_caption("Columnas casi constantes (moda ≥ 99%)")
)
display(estilo_tabla(reglas_calidad, "Porcentaje").set_caption("Reglas de calidad"))



## 2.4 Perfil estadístico numérico y valores extremos



In [ ]:
numericas_analizables = [
    c for c in columnas_numericas
    if c not in IDENTIFICADORES and datos[c].notna().any()
]

perfil_numerico = datos[numericas_analizables].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T
perfil_numerico["faltantes"] = datos[numericas_analizables].isna().sum()
perfil_numerico["porcentaje_faltante"] = (
    100 * perfil_numerico["faltantes"] / len(datos)
).round(2)
perfil_numerico["asimetria"] = datos[numericas_analizables].skew(numeric_only=True)
perfil_numerico = perfil_numerico.reset_index(names="Variable")

display(estilo_tabla(perfil_numerico.head(40), "porcentaje_faltante").set_caption("Perfil numérico"))
perfil_numerico.to_csv(CARPETA_SALIDA / "02_perfil_numerico.csv", sep=";", index=False)



In [ ]:
filas_atipicos = []
for columna in numericas_analizables:
    serie = datos[columna].dropna()
    if len(serie) < 4:
        continue
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    inferior, superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    cantidad = int(((serie < inferior) | (serie > superior)).sum())
    filas_atipicos.append(
        (columna, inferior, superior, cantidad, round(100 * cantidad / len(serie), 2))
    )

tabla_atipicos = pd.DataFrame(
    filas_atipicos,
    columns=["Variable", "LimiteInferiorIQR", "LimiteSuperiorIQR", "CantidadAtipicos", "PorcentajeAtipicos"],
).sort_values("PorcentajeAtipicos", ascending=False)
display(estilo_tabla(tabla_atipicos.head(30), "PorcentajeAtipicos").set_caption("Top de valores extremos por regla IQR"))
tabla_atipicos.to_csv(CARPETA_SALIDA / "03_atipicos_iqr.csv", sep=";", index=False)



## 2.5 Perfil categórico



In [ ]:
filas_categoricas = []
for columna in CATEGORICAS:
    conteo = datos[columna].value_counts(dropna=False)
    moda = conteo.index[0] if len(conteo) else pd.NA
    frecuencia = int(conteo.iloc[0]) if len(conteo) else 0
    filas_categoricas.append(
        {
            "Variable": columna,
            "ValoresUnicos": datos[columna].nunique(dropna=True),
            "Moda": moda,
            "FrecuenciaModa": frecuencia,
            "PorcentajeModa": round(100 * frecuencia / len(datos), 2),
            "Faltantes": int(datos[columna].isna().sum()),
            "PorcentajeFaltante": round(100 * datos[columna].isna().mean(), 2),
        }
    )

perfil_categorico = pd.DataFrame(filas_categoricas).sort_values("ValoresUnicos", ascending=False)
display(estilo_tabla(perfil_categorico, "PorcentajeFaltante").set_caption("Perfil categórico"))
perfil_categorico.to_csv(CARPETA_SALIDA / "04_perfil_categorico.csv", sep=";", index=False)



## 2.6 Indicadores del negocio y distribución de mora



In [ ]:
saldo_total = datos["SaldoCreditoBs"].sum(min_count=1)
mora_valida = datos["MaxDiasMoraMes"].dropna()
kpis = pd.DataFrame(
    {
        "Indicador": [
            "Créditos", "Clientes", "Saldo total Bs", "Saldo promedio Bs", "Saldo mediano Bs",
            "Mora ≥ 1 día", "Mora ≥ 30 días", "Mora ≥ 90 días",
        ],
        "Cantidad": [
            len(datos), datos["ClienteKey"].nunique(), saldo_total,
            datos["SaldoCreditoBs"].mean(), datos["SaldoCreditoBs"].median(),
            int((mora_valida >= 1).sum()), int((mora_valida >= 30).sum()), int((mora_valida >= 90).sum()),
        ],
        "Porcentaje": [
            100, 100, np.nan, np.nan, np.nan,
            100 * (mora_valida >= 1).mean(),
            100 * (mora_valida >= 30).mean(),
            100 * (mora_valida >= 90).mean(),
        ],
    }
)
display(estilo_tabla(kpis, "Porcentaje").set_caption("Indicadores principales"))

distribucion_mora = datos["BandaMora"].value_counts().reindex(etiquetas_mora + ["Sin dato"], fill_value=0)
grafico_barras(distribucion_mora, "Distribución por banda de mora", "03_bandas_mora.png")



In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(15, 5))
ejes[0].hist(datos["EdadCliente"].dropna(), bins=30, color=LILA, edgecolor=BLANCO)
ejes[0].axvline(datos["EdadCliente"].median(), color=MAGENTA, linestyle="--", label="Mediana")
ejes[0].set_title("Distribución de edad")
ejes[0].set_xlabel("Edad")
ejes[0].legend()

saldo_p99 = datos["SaldoCreditoBs"].quantile(0.99)
ejes[1].hist(datos.loc[datos["SaldoCreditoBs"] <= saldo_p99, "SaldoCreditoBs"].dropna(), bins=35, color=AZUL)
ejes[1].set_title("Saldo de crédito hasta percentil 99")
ejes[1].set_xlabel("Bolivianos")
for eje in ejes:
    eje.grid(axis="y")
guardar_figura("04_distribuciones_edad_saldo.png")



## 2.7 Concentración de cartera y mora por segmentos



In [ ]:
def resumen_segmento(columna: str, minimo: int = 100) -> pd.DataFrame:
    tabla = (
        datos.groupby(columna, dropna=False)
        .agg(
            Creditos=("CarteraKey", "size"),
            Clientes=("ClienteKey", "nunique"),
            SaldoTotalBs=("SaldoCreditoBs", "sum"),
            SaldoPromedioBs=("SaldoCreditoBs", "mean"),
            Mora30Cantidad=("Mora30", "sum"),
            Mora30Tasa=("Mora30", "mean"),
        )
        .reset_index()
    )
    tabla = tabla[tabla["Creditos"] >= minimo].copy()
    tabla["ParticipacionSaldo"] = 100 * tabla["SaldoTotalBs"] / tabla["SaldoTotalBs"].sum()
    tabla["Mora30Tasa"] = 100 * tabla["Mora30Tasa"]
    return tabla.sort_values("SaldoTotalBs", ascending=False)


resumenes_segmentos = {}
for variable in [
    "DepartamentoCliente", "SegmentoCliente", "TipoCredito", "SectorEconomicoCredito",
    "GeneroCliente", "EsClienteDigital", "ProductoCartera",
]:
    resumenes_segmentos[variable] = resumen_segmento(variable)
    display(
        estilo_tabla(resumenes_segmentos[variable].head(15), "Mora30Tasa")
        .set_caption(f"Cartera y mora por {variable}")
    )



In [ ]:
for variable in ["DepartamentoCliente", "SegmentoCliente", "TipoCredito", "SectorEconomicoCredito"]:
    tabla = resumenes_segmentos[variable].head(12).sort_values("SaldoTotalBs")
    fig, ax = plt.subplots(figsize=(12, max(5, len(tabla) * 0.45)))
    barras = ax.barh(tabla[variable].astype(str), tabla["SaldoTotalBs"], color=LILA)
    ax.set_title(f"Saldo total por {variable}")
    ax.set_xlabel("Bolivianos")
    ax.grid(axis="x")
    guardar_figura(f"05_saldo_{variable}.png")



## 2.8 Cobertura de fuentes y actividad transaccional



In [ ]:
cobertura = pd.DataFrame(
    {
        "Fuente": INDICADORES_COBERTURA,
        "CantidadConDato": [int((datos[c] == 1).sum()) for c in INDICADORES_COBERTURA],
    }
)
cobertura["CantidadSinDato"] = len(datos) - cobertura["CantidadConDato"]
cobertura["PorcentajeCobertura"] = 100 * cobertura["CantidadConDato"] / len(datos)
display(estilo_tabla(cobertura, "PorcentajeCobertura").set_caption("Cobertura de fuentes opcionales"))

fig, ax = plt.subplots(figsize=(10, 5))
barras = ax.bar(cobertura["Fuente"], cobertura["PorcentajeCobertura"], color=PALETA[:4])
ax.bar_label(barras, labels=[f"{v:.1f}%" for v in cobertura["PorcentajeCobertura"]], padding=3)
ax.set_ylim(0, 110)
ax.set_ylabel("Porcentaje")
ax.set_title("Cobertura de las fuentes complementarias")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y")
guardar_figura("06_cobertura_fuentes.png")



In [ ]:
transacciones = datos.loc[datos["TieneTransaccionesMes"].eq(1), [
    "CantidadTrx", "MontoTotalTrxBs", "CantidadDepositos", "CantidadRetiros", "FlujoNetoBs"
]].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).T
display(estilo_tabla(transacciones.reset_index(names="Variable")).set_caption("Actividad transaccional donde existe cobertura"))



## 2.9 Vista por cliente y concentración

Una persona puede tener varias operaciones. Esta vista evita interpretar
cada crédito como si fuera un cliente distinto.



In [ ]:
clientes = (
    datos.groupby("ClienteKey", dropna=False)
    .agg(
        Creditos=("CarteraKey", "nunique"),
        SaldoTotalBs=("SaldoCreditoBs", "sum"),
        MaxMoraMes=("MaxDiasMoraMes", "max"),
        Edad=("EdadCliente", "first"),
        Departamento=("DepartamentoCliente", "first"),
        Segmento=("SegmentoCliente", "first"),
    )
    .reset_index()
)
clientes["Mora30"] = (clientes["MaxMoraMes"] >= 30).astype("int8")
clientes = clientes.sort_values("SaldoTotalBs", ascending=False)
clientes["ParticipacionAcumuladaSaldo"] = 100 * clientes["SaldoTotalBs"].cumsum() / clientes["SaldoTotalBs"].sum()

concentracion = pd.DataFrame(
    {
        "Indicador": ["Clientes", "Créditos por cliente promedio", "Clientes con más de un crédito", "Mora30 por cliente"],
        "Valor": [
            len(clientes), clientes["Creditos"].mean(), int((clientes["Creditos"] > 1).sum()),
            100 * clientes["Mora30"].mean(),
        ],
    }
)
display(estilo_tabla(concentracion).set_caption("Resumen por cliente"))



## 2.10 Correlaciones numéricas

Se utiliza Spearman porque las variables financieras presentan asimetría y
valores extremos. Para controlar memoria se limita el cálculo a 80.000 filas.



In [ ]:
variables_correlacion = [
    "EdadCliente", "AntiguedadClienteMeses", "PatrimonioClienteBs", "ActivosClienteBs",
    "SaldoCreditoBs", "MontoDesembolsadoBs", "TasaEfectivaCredito", "PlazoCreditoMeses",
    "CantidadReprogramaciones", "MontoGarantiaTotalBs", "MaxDiasMoraMes",
    "CantidadTrx3M", "MontoTotalBs3M", "CantidadTrx6M", "MontoTotalBs6M",
    "CantidadTrx12M", "MontoTotalBs12M", "MesesConHistoria",
]
variables_correlacion = [c for c in variables_correlacion if datos[c].notna().any()]
muestra_correlacion = datos[variables_correlacion]
if len(muestra_correlacion) > 80_000:
    muestra_correlacion = muestra_correlacion.sample(80_000, random_state=SEMILLA)
matriz_correlacion = muestra_correlacion.corr(method="spearman")

fig, ax = plt.subplots(figsize=(14, 11))
imagen = ax.imshow(matriz_correlacion, cmap=CMAP, vmin=-1, vmax=1)
ax.set_xticks(range(len(matriz_correlacion.columns)))
ax.set_xticklabels(matriz_correlacion.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(matriz_correlacion.index)))
ax.set_yticklabels(matriz_correlacion.index, fontsize=8)
fig.colorbar(imagen, ax=ax, label="Correlación Spearman")
ax.set_title("Matriz de correlaciones")
guardar_figura("07_correlaciones_spearman.png")



## 4. Modelado diagnóstico exploratorio

Se ajusta una regresión logística numérica implementada con NumPy. Se divide
por `ClienteKey`, de modo que un cliente no aparezca simultáneamente en
entrenamiento y prueba. Se excluyen variables contemporáneas de mora,
calificaciones y previsiones. Esto **no** convierte un solo mes en un modelo
de predicción futura; solo mide separabilidad dentro del mismo corte.



In [ ]:
VARIABLES_MODELO = [
    "EdadCliente", "AntiguedadClienteMeses", "NumeroHijosCliente", "PatrimonioClienteBs",
    "ActivosClienteBs", "VentasAnualesClienteBs", "InfocredMontoDeudaTotalBs",
    "InfocredCuotaBs", "InfocredCantDeudasOtrasEntidades", "TasaEfectivaCredito",
    "PlazoCreditoMeses", "CantidadReprogramaciones", "SaldoCreditoBs",
    "MontoDesembolsadoBs", "MontoGarantiaTotalBs", "AntiguedadCreditoMeses",
    "CantidadTrx3M", "MontoTotalBs3M", "CantidadTrx6M", "MontoTotalBs6M",
    "CantidadTrx12M", "MontoTotalBs12M", "MesesConHistoria",
]
VARIABLES_MODELO = [c for c in VARIABLES_MODELO if datos[c].notna().any()]


def sigmoide(z):
    z = np.clip(z, -35, 35)
    return 1.0 / (1.0 + np.exp(-z))


def auc_binario(y, puntaje):
    y = np.asarray(y, dtype=int)
    puntaje = np.asarray(puntaje, dtype=float)
    positivos, negativos = int(y.sum()), int((1 - y).sum())
    if positivos == 0 or negativos == 0:
        return np.nan
    rangos = pd.Series(puntaje).rank(method="average").to_numpy()
    return (rangos[y == 1].sum() - positivos * (positivos + 1) / 2) / (positivos * negativos)


def metricas_clasificacion(y, puntaje, umbral=0.5):
    y = np.asarray(y, dtype=int)
    pred = (np.asarray(puntaje) >= umbral).astype(int)
    vp = int(((pred == 1) & (y == 1)).sum())
    vn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    precision = vp / (vp + fp) if vp + fp else 0
    recall = vp / (vp + fn) if vp + fn else 0
    especificidad = vn / (vn + fp) if vn + fp else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    return pd.DataFrame(
        {
            "Metrica": ["AUC ROC", "Precision", "Recall", "Especificidad", "F1", "VP", "VN", "FP", "FN"],
            "Valor": [auc_binario(y, puntaje), precision, recall, especificidad, f1, vp, vn, fp, fn],
        }
    )


resultado_modelo = None
coeficientes_modelo = None
if EJECUTAR_MODELO_DIAGNOSTICO:
    base_modelo = datos.loc[datos["Mora30"].notna(), ["ClienteKey", "Mora30"] + VARIABLES_MODELO].copy()
    base_modelo[VARIABLES_MODELO] = base_modelo[VARIABLES_MODELO].replace([np.inf, -np.inf], np.nan)
    y_total = base_modelo["Mora30"].astype(int).to_numpy()

    hashes = pd.util.hash_pandas_object(base_modelo["ClienteKey"].astype("string"), index=False)
    mascara_prueba = hashes.mod(5).eq(0).to_numpy()
    indices_entrena = np.flatnonzero(~mascara_prueba)
    indices_prueba = np.flatnonzero(mascara_prueba)

    rng = np.random.default_rng(SEMILLA)
    if len(indices_entrena) > 120_000:
        indices_entrena = rng.choice(indices_entrena, 120_000, replace=False)
    if len(indices_prueba) > 50_000:
        indices_prueba = rng.choice(indices_prueba, 50_000, replace=False)

    x_entrena_df = base_modelo.iloc[indices_entrena][VARIABLES_MODELO].copy()
    x_prueba_df = base_modelo.iloc[indices_prueba][VARIABLES_MODELO].copy()
    y_entrena = y_total[indices_entrena]
    y_prueba = y_total[indices_prueba]

    medianas = x_entrena_df.median()
    limites_bajos = x_entrena_df.quantile(0.01)
    limites_altos = x_entrena_df.quantile(0.99)
    medias = x_entrena_df.clip(limites_bajos, limites_altos, axis=1).fillna(medianas).mean()
    desvios = x_entrena_df.clip(limites_bajos, limites_altos, axis=1).fillna(medianas).std().replace(0, 1)

    def transformar_modelo(tabla):
        x = tabla.clip(limites_bajos, limites_altos, axis=1).fillna(medianas)
        x = ((x - medias) / desvios).to_numpy(dtype="float64")
        return np.column_stack([np.ones(len(x)), x])

    x_entrena = transformar_modelo(x_entrena_df)
    x_prueba = transformar_modelo(x_prueba_df)

    if len(np.unique(y_entrena)) < 2 or len(np.unique(y_prueba)) < 2:
        print("Modelo omitido: la variable objetivo no contiene ambas clases en la partición.")
    else:
        beta = np.zeros(x_entrena.shape[1])
        positivos = max(int(y_entrena.sum()), 1)
        negativos = max(int((1 - y_entrena).sum()), 1)
        pesos = np.where(y_entrena == 1, len(y_entrena) / (2 * positivos), len(y_entrena) / (2 * negativos))
        tasa_aprendizaje = 0.08
        regularizacion = 0.01

        for _ in range(300):
            probabilidades = sigmoide(x_entrena @ beta)
            gradiente = x_entrena.T @ (pesos * (probabilidades - y_entrena)) / pesos.sum()
            gradiente[1:] += regularizacion * beta[1:]
            beta -= tasa_aprendizaje * gradiente

        puntaje_prueba = sigmoide(x_prueba @ beta)
        resultado_modelo = metricas_clasificacion(y_prueba, puntaje_prueba)
        coeficientes_modelo = pd.DataFrame(
            {"Variable": ["Intercepto"] + VARIABLES_MODELO, "Coeficiente": beta}
        )
        coeficientes_modelo["ImpactoAbsoluto"] = coeficientes_modelo["Coeficiente"].abs()
        coeficientes_modelo = coeficientes_modelo.sort_values("ImpactoAbsoluto", ascending=False)

        display(estilo_tabla(resultado_modelo).set_caption("Evaluación fuera de muestra por cliente"))
        display(estilo_tabla(coeficientes_modelo.head(20), "ImpactoAbsoluto").set_caption("Variables numéricas con mayor efecto diagnóstico"))

        orden = np.argsort(-puntaje_prueba)
        y_ordenado = y_prueba[orden]
        tpr = np.r_[0, np.cumsum(y_ordenado) / y_ordenado.sum()]
        fpr = np.r_[0, np.cumsum(1 - y_ordenado) / (1 - y_ordenado).sum()]
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.plot(fpr, tpr, color=FUCSIA, linewidth=3, label=f"AUC = {auc_binario(y_prueba, puntaje_prueba):.3f}")
        ax.plot([0, 1], [0, 1], color=AZUL_OSCURO, linestyle="--")
        ax.set_xlabel("Tasa de falsos positivos")
        ax.set_ylabel("Tasa de verdaderos positivos")
        ax.set_title("Curva ROC — modelo diagnóstico")
        ax.legend()
        ax.grid()
        guardar_figura("08_curva_roc_modelo_diagnostico.png")



## 5. Evaluación y conclusiones automáticas



In [ ]:
tasa_mora30 = 100 * datos["Mora30"].mean()
peor_faltante = tabla_faltantes.iloc[0]
coberturas_texto = ", ".join(
    f"{fila.Fuente}: {fila.PorcentajeCobertura:.1f}%" for fila in cobertura.itertuples()
)

hallazgos = f"""
### Hallazgos del corte {periodos[0]}

- Se analizaron **{len(datos):,} créditos**, **{datos['ClienteKey'].nunique():,} clientes** y **{len(COLUMNAS)} variables**.
- El saldo total observado es **Bs {saldo_total:,.2f}**.
- La tasa descriptiva de mora de 30 días o más es **{tasa_mora30:.2f}%** sobre registros con mora disponible.
- La columna con mayor faltante es **{peor_faltante['Columna']}**, con **{peor_faltante['CantidadFaltante']:,.0f} registros ({peor_faltante['PorcentajeFaltante']:.2f}%)**.
- Existen **{len(columnas_vacias)} columnas completamente vacías** y **{len(columnas_constantes)} constantes**; deben excluirse de cualquier modelo.
- Se encontraron **{duplicados_fila:,} duplicados exactos** y **{duplicados_clave:,} filas asociadas a claves de cartera repetidas**.
- Cobertura de fuentes: {coberturas_texto}.
- Los valores extremos no deben eliminarse automáticamente: primero se debe confirmar si son operaciones grandes válidas o errores.
- Con un solo mes, las asociaciones encontradas son contemporáneas. Para predecir mora se debe usar información de T y observar el resultado en T+1 o un horizonte posterior.
"""
display(Markdown(hallazgos))



## 6. Entrega y exportación

Se exportan las tablas principales y un manifiesto. Los gráficos ya fueron
guardados durante la ejecución. El dataset limpio completo se exporta solo si
`EXPORTAR_DATOS_LIMPIOS = True` para evitar duplicar un archivo muy grande.



In [ ]:
tablas_salida = {
    "00_resumen_general": resumen_general,
    "01_faltantes_por_columna": tabla_faltantes,
    "02_reglas_calidad": reglas_calidad,
    "03_perfil_numerico": perfil_numerico,
    "04_atipicos_iqr": tabla_atipicos,
    "05_perfil_categorico": perfil_categorico,
    "06_kpis": kpis,
    "07_cobertura": cobertura,
    "08_clientes": clientes.head(100_000),
}
for variable, tabla in resumenes_segmentos.items():
    tablas_salida[f"segmento_{variable}"] = tabla
if resultado_modelo is not None:
    tablas_salida["modelo_metricas"] = resultado_modelo
if coeficientes_modelo is not None:
    tablas_salida["modelo_coeficientes"] = coeficientes_modelo

for nombre, tabla in tablas_salida.items():
    tabla.to_csv(CARPETA_SALIDA / f"{nombre}.csv", sep=";", index=False, encoding="utf-8-sig")

if EXPORTAR_DATOS_LIMPIOS:
    datos.to_csv(
        CARPETA_SALIDA / "datos22_limpio.csv", sep=";", index=False, encoding="utf-8-sig"
    )

manifiesto = {
    "archivo_fuente": str(ruta_csv),
    "mes": str(periodos[0]),
    "filas": int(len(datos)),
    "columnas_originales": int(len(COLUMNAS)),
    "duplicados_exactos": duplicados_fila,
    "columnas_vacias": columnas_vacias,
    "columnas_constantes": columnas_constantes,
    "salida": str(CARPETA_SALIDA.resolve()),
    "limitacion": "Un mes: el modelo es diagnóstico, no predictivo futuro.",
}
with open(CARPETA_SALIDA / "manifiesto.json", "w", encoding="utf-8") as archivo:
    json.dump(manifiesto, archivo, ensure_ascii=False, indent=2)

print(f"Análisis terminado. Resultados guardados en: {CARPETA_SALIDA.resolve()}")



## Recomendaciones para el siguiente ciclo

1. Incorporar al menos 12 cortes mensuales con el mismo diccionario.
2. Definir el objetivo futuro, por ejemplo `Mora30` en T+1.
3. Excluir del entrenamiento variables conocidas después del cierre o que
   revelen directamente mora, calificación y previsión del objetivo.
4. Separar entrenamiento y validación por tiempo, no aleatoriamente.
5. Confirmar con negocio las columnas vacías de eventos y las coberturas
   parciales antes de imputar.
6. Documentar reglas para valores negativos, extremos y códigos desconocidos.
7. Monitorear mensualmente calidad, estabilidad poblacional y desempeño.
